# XYZ → RDF/PDF Simulation

이 노트북은 XYZ 파일로부터 **Radial Distribution Function (RDF)** 과 **Pair Distribution Function (PDF)** 를 계산하고 시각화합니다.

## 실행 순서
1. **셀 1** - 라이브러리 임포트 및 함수 정의
2. **셀 2** - XYZ 파일 경로 및 파라미터 설정
3. **셀 3** - 시뮬레이션 실행
4. **셀 4** - 결과 시각화 (PDF, RDF, 회절 패턴)
5. **셀 5** - (선택) Damping 파라미터 재조정 후 빠른 재계산

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import RDF_Package

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 12

# ── 내부 함수 정의 (simulation.py와 동일) ─────────────────────────────────────

def atom2number(atom_string, mode=1):
    atom_dict = {
        'H':1,'He':2,'Li':3,'Be':4,'B':5,'C':6,'N':7,'O':8,'F':9,'Ne':10,
        'Na':11,'Mg':12,'Al':13,'Si':14,'P':15,'S':16,'Cl':17,'Ar':18,'K':19,'Ca':20,
        'Sc':21,'Ti':22,'V':23,'Cr':24,'Mn':25,'Fe':26,'Co':27,'Ni':28,'Cu':29,'Zn':30,
        'Ga':31,'Ge':32,'As':33,'Se':34,'Br':35,'Kr':36,'Rb':37,'Sr':38,'Y':39,'Zr':40,
        'Nb':41,'Mo':42,'Tc':43,'Ru':44,'Rh':45,'Pd':46,'Ag':47,'Cd':48,'In':49,'Sn':50,
        'Sb':51,'Te':52,'I':53,'Xe':54,'Cs':55,'Ba':56,'La':57,'Ce':58,'Pr':59,'Nd':60,
        'Pm':61,'Sm':62,'Eu':63,'Gd':64,'Tb':65,'Dy':66,'Ho':67,'Er':68,'Tm':69,'Yb':70,
        'Lu':71,'Hf':72,'Ta':73,'W':74,'Re':75,'Os':76,'Ir':77,'Pt':78,'Au':79,'Hg':80,
        'Tl':81,'Pb':82,'Bi':83,'Po':84,'At':85,'Rn':86,'Fr':87,'Ra':88,'Ac':89,'Th':90,
        'Pa':91,'U':92,'Np':93,'Pu':94,'Am':95,'Cm':96,'Bk':97,'Cf':98,'Es':99,'Fm':100,
        'Md':101,'No':102,'Lr':103,'Rf':104,'Db':105,'Sg':106,'Bh':107,'Hs':108,'Mt':109,
        'Ds':110,'Rg':111,'Uub':112
    }
    if mode == 1:
        return atom_dict.get(atom_string, 0)
    elif mode == 2:
        for key, val in atom_dict.items():
            if val == atom_string:
                return key
    return 0


def read_xyz_file(path):
    """XYZ 파일을 읽어 원자 종류, 좌표, 박스 크기를 반환합니다."""
    aType = []
    coords = np.zeros((1, 3))
    with open(path, 'r') as f:
        line_data = f.readline()
        while line_data != '':
            try:
                int(line_data)
                line_data = f.readline()
                line_data = f.readline()
                continue
            except Exception:
                pass
            line_data = line_data.split()
            aType.append(atom2number(line_data[0]))
            for i, item in enumerate(line_data[1:]):
                line_data[i + 1] = float(item)
            coords = np.vstack((coords, np.array(line_data[1:])))
            line_data = f.readline()
        coords = coords[1:, :]

    mn = np.array([
        [np.amax(coords[:, 0]) - np.amin(coords[:, 0]), 0, 0],
        [0, np.amax(coords[:, 1]) - np.amin(coords[:, 1]), 0],
        [0, 0, np.amax(coords[:, 2]) - np.amin(coords[:, 2])]
    ])
    return np.array(aType), coords, mn


def compute_pdf(coords, aType, atomKinds, dr, Nz, Nr, Npdf, Mm, s, f):
    rMax = dr * Nr
    PDF = np.zeros((Nr, Npdf))
    phai = np.zeros((np.size(s), Npdf))
    s[0] = 0.0000001
    pdfCount = -1
    for i in np.arange(Nz):
        for j in np.arange(i, Nz):
            pdfCount += 1
            index1 = np.where(aType == atomKinds[i])
            p1 = coords[index1[0]]
            index2 = np.where(aType == atomKinds[j])
            p2 = coords[index2[0]]
            atom1_num = np.shape(p1)[0]
            atom2_num = np.shape(p2)[0]
            for m in np.arange(atom1_num):
                l = m if atomKinds[i] == atomKinds[j] else 0
                for n in np.arange(l, atom2_num):
                    d = np.sqrt(np.sum(np.power(p1[m, :] - p2[n, :], 2)))
                    if 0 < d < rMax:
                        rol = abs(int(d / dr + 0.5))
                        PDF[rol - 1, pdfCount] += 1
            fsqr = np.multiply(f[:, i], f[:, j])
            for k in np.arange(1, Nr):
                phai[:, pdfCount] += PDF[k, pdfCount] * np.multiply(fsqr, np.sinc(2 * s * dr * k))
    return PDF, 2 * phai


def count_pdf_function(coords, aType, Mm, bin=1, dr=0.02, max_angle=10, califactor=0.00111625):
    rMax = np.linalg.norm(np.diag(Mm))
    r = np.arange(0, rMax + dr, dr)
    Nr = np.size(r)
    atom_kinds = np.unique(aType)
    atom_number = np.size(aType)
    Nz = np.size(atom_kinds)
    Npdf = int(Nz + Nz * (Nz - 1) / 2)
    califactor = califactor * bin
    n = int(max_angle / califactor)
    s = np.arange(n) * califactor
    s_sqr = np.power(s, 2)
    f = np.zeros((len(s), Nz))
    fmean = np.zeros(len(s))
    fsqrmean = np.zeros(len(s))

    for i in np.arange(Nz):
        fit_para = RDF_Package.ref_atom_para(atom_kinds[i])
        f[:, i] = (
            fit_para[0, 0] * np.power((s_sqr + fit_para[0, 1]), -1) +
            fit_para[0, 2] * np.power((s_sqr + fit_para[0, 3]), -1) +
            fit_para[1, 0] * np.power((s_sqr + fit_para[1, 1]), -1) +
            fit_para[1, 2] * np.exp(-1 * fit_para[1, 3] * s_sqr) +
            fit_para[2, 0] * np.exp(-1 * fit_para[2, 1] * s_sqr) +
            fit_para[2, 2] * np.exp(-1 * fit_para[2, 3] * s_sqr)
        )
        composition = np.size(np.where(aType == atom_kinds[i])) / atom_number
        fmean += composition * f[:, i]
        fsqrmean += composition * np.power(f[:, i], 2)

    PDF, phai = compute_pdf(coords, aType, atom_kinds, dr, Nz, Nr, Npdf, Mm, s, f)
    diftot = np.sum(phai, 1) + atom_number * fsqrmean
    return phai, diftot, fsqrmean, atom_number, s, PDF, atom_kinds


def ftg(r, s, dif, be, en):
    ds = s[1] - s[0]
    Gp = np.zeros(np.size(r))
    for i in np.arange(np.size(r)):
        dif_temp = dif[be:en + 1]
        s_temp = s[be:en + 1]
        sin_temp = np.sin(s_temp * r[i] * 2 * np.pi)
        Gp[i] = np.sum(np.multiply(dif_temp, sin_temp)) * ds * 8 * np.pi
    return Gp


def Gc_rdf(atom_number, s, diff, fmeansqr, maxrange=10, step=0.01,
           be=0.3, en=3, E=0.25, H=0.015, norm_factor=1):
    be = int(be / (s[1] - s[0]))
    en = int(en / (s[1] - s[0]))
    r = np.arange(0, maxrange, step)
    diffs = diff * 2
    damp = np.exp(-1 * E * np.power(s, 2))
    Npdf = np.shape(diff)[1]
    G = np.zeros((np.size(r), Npdf))
    for i in np.arange(Npdf):
        diffs[:, i] = np.divide(
            np.multiply(diff[:, i], s), fmeansqr * atom_number
        ) * norm_factor
        diffs[:, i] = np.multiply(diffs[:, i], damp)
        G[:, i] = np.multiply(
            ftg(r, s, diffs[:, i], be - 1, en),
            np.exp(-1 * H * np.power(r, 2))
        )
    Gtot = np.sum(G, 1)
    return G, Gtot, r, diffs


def simulation_with_xyz(path, parameter):
    """XYZ 파일에서 PDF/RDF를 계산합니다. 결과를 반환하고 recal_para도 함께 반환합니다."""
    aType, coords, Mm = read_xyz_file(path)
    phai, diftot, fmeansqr, atom_number, s, PDF, atom_kinds = count_pdf_function(
        coords, aType, Mm,
        max_angle=parameter['maxangle'],
        califactor=parameter['cali']
    )
    recal_para = {
        'atom_number': atom_number,
        's': s,
        'phai': phai,
        'fmeansqr': fmeansqr,
        'PDF': PDF,
        'atom_kinds': atom_kinds,
        'diftot': diftot
    }
    G, Gtot, r, diffs = Gc_rdf(
        atom_number, s, phai, fmeansqr,
        maxrange=parameter['max_range'],
        step=parameter['step_length'],
        be=parameter['window_start'],
        en=parameter['window_end'],
        E=parameter['pdf_damp'],
        H=parameter['rdf_damp']
    )
    atom = [atom2number(item, mode=2) for item in atom_kinds]
    return PDF, G, Gtot, r, atom, s, diffs, diftot, recal_para


def recal_damp(recal_para, parameter):
    """XYZ 재로딩 없이 damping 파라미터만 바꿔 빠르게 재계산합니다."""
    G, Gtot, r, diffs = Gc_rdf(
        recal_para['atom_number'], recal_para['s'], recal_para['phai'], recal_para['fmeansqr'],
        maxrange=parameter['max_range'],
        step=parameter['step_length'],
        be=parameter['window_start'],
        en=parameter['window_end'],
        E=parameter['pdf_damp'],
        H=parameter['rdf_damp']
    )
    atom = [atom2number(item, mode=2) for item in recal_para['atom_kinds']]
    return recal_para['PDF'], G, Gtot, r, atom, recal_para['s'], diffs, recal_para['diftot'], recal_para


print('함수 정의 완료.')

## Step 1 — 파일 경로 및 파라미터 설정

| 파라미터 | 설명 |
|---|---|
| `xyz_path` | XYZ 파일 경로 |
| `cali` | 캘리브레이션 팩터 (1/Å per pixel) |
| `maxangle` | 최대 산란각 (1/Å) |
| `window_start` | Fourier transform 시작 q 값 (1/Å) |
| `window_end` | Fourier transform 끝 q 값 (1/Å) |
| `pdf_damp` | PDF damping 강도 (E) |
| `rdf_damp` | RDF Gaussian damping 강도 (H) |
| `max_range` | G(r) 계산 최대 r 범위 (Å) |
| `step_length` | G(r) r 간격 (Å) |

In [ ]:
# ── 여기를 수정하세요 ─────────────────────────────────────────────────────────

xyz_path = 'sample.xyz'        # XYZ 파일 경로

parameter = {
    'cali'         : 0.00111625,  # 캘리브레이션 팩터
    'maxangle'     : 10,          # 최대 산란각 (1/Å)
    'window_start' : 0.3,         # Fourier window 시작 q (1/Å)
    'window_end'   : 3.0,         # Fourier window 끝 q (1/Å)
    'pdf_damp'     : 0.25,        # PDF damping 강도
    'rdf_damp'     : 0.015,       # RDF damping 강도
    'max_range'    : 10.0,        # G(r) 최대 r (Å)
    'step_length'  : 0.01,        # G(r) r 간격 (Å)
}

# ─────────────────────────────────────────────────────────────────────────────
print('파라미터 설정 완료:')
for k, v in parameter.items():
    print(f'  {k:15s} = {v}')

## Step 2 — 시뮬레이션 실행

> XYZ 파일 크기에 따라 수 분이 걸릴 수 있습니다.

In [ ]:
import time

t0 = time.time()
PDF, G, Gtot, r, atom, s, diffs, diftot, recal_para = simulation_with_xyz(xyz_path, parameter)
elapsed = time.time() - t0

print(f'계산 완료  ({elapsed:.1f} 초)')
print(f'원자 종류  : {atom}')
print(f'원자 수    : {recal_para["atom_number"]}')
print(f'PDF 쌍 수  : {PDF.shape[1]}')

## Step 3 — 결과 시각화

In [ ]:
# ── Pair Distribution Function (PDF) ─────────────────────────────────────────
atom_kinds = recal_para['atom_kinds']
Nz = len(atom_kinds)
labels = []
for i in range(Nz):
    for j in range(i, Nz):
        labels.append(f'{atom2number(atom_kinds[i], mode=2)}-{atom2number(atom_kinds[j], mode=2)}')

dr = 0.02
r_pdf = np.arange(PDF.shape[0]) * dr

n_pairs = PDF.shape[1]
fig, axes = plt.subplots(n_pairs, 1, figsize=(10, 4 * n_pairs), sharex=True)
if n_pairs == 1:
    axes = [axes]

for idx, ax in enumerate(axes):
    ax.bar(r_pdf, PDF[:, idx], width=dr * 0.9, color='steelblue', alpha=0.8)
    ax.set_ylabel('Counts')
    ax.set_title(f'PDF — {labels[idx]}')

axes[-1].set_xlabel('r (Å)')
plt.suptitle('Pair Distribution Functions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── G(r) — Total RDF ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

axes[0].plot(r, Gtot, color='navy', lw=1.5)
axes[0].axhline(0, color='gray', lw=0.8, ls='--')
axes[0].set_xlabel('r (Å)')
axes[0].set_ylabel('G(r)')
axes[0].set_title('Total G(r) — RDF')

# 각 쌍별 G(r)
for idx in range(G.shape[1]):
    axes[1].plot(r, G[:, idx], lw=1.5, label=labels[idx])
axes[1].axhline(0, color='gray', lw=0.8, ls='--')
axes[1].set_xlabel('r (Å)')
axes[1].set_ylabel('G(r)')
axes[1].set_title('Partial G(r) — 원자 쌍별')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 회절 강도 I(s) ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

axes[0].plot(s, diftot, color='darkred', lw=1.5)
axes[0].set_xlabel('s (1/Å)')
axes[0].set_ylabel('I(s)')
axes[0].set_title('Total Diffraction Intensity I(s)')

for idx in range(diffs.shape[1]):
    axes[1].plot(s, diffs[:, idx], lw=1.2, label=labels[idx])
axes[1].axhline(0, color='gray', lw=0.8, ls='--')
axes[1].set_xlabel('s (1/Å)')
axes[1].set_ylabel('s·i(s)')
axes[1].set_title('Reduced Structure Function s·i(s) (damped)')
axes[1].legend()

# Fourier window 표시
for ax in axes:
    ax.axvline(parameter['window_start'], color='green', lw=1, ls=':', label='window')
    ax.axvline(parameter['window_end'],   color='green', lw=1, ls=':')

plt.tight_layout()
plt.show()

## Step 4 — (선택) Damping 파라미터 빠른 재조정

XYZ 파일을 다시 읽지 않고 damping 값만 바꿔 G(r)을 빠르게 재계산합니다.

In [ ]:
# ── 재조정할 파라미터만 수정하세요 ───────────────────────────────────────────
new_parameter = {
    'window_start' : 0.3,
    'window_end'   : 4.0,   # ← 변경
    'pdf_damp'     : 0.15,  # ← 변경
    'rdf_damp'     : 0.010, # ← 변경
    'max_range'    : 10.0,
    'step_length'  : 0.01,
}

_, G2, Gtot2, r2, atom2, s2, diffs2, diftot2, _ = recal_damp(recal_para, new_parameter)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(r,  Gtot,  color='navy',  lw=1.5, alpha=0.6, label='Original')
ax.plot(r2, Gtot2, color='coral', lw=1.5, ls='--',   label='Re-calculated')
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('r (Å)')
ax.set_ylabel('G(r)')
ax.set_title('G(r) 비교 — Damping 파라미터 변경 전/후')
ax.legend()
plt.tight_layout()
plt.show()